# PROCESADO DE IMAGNES FACEFORENSICS++ C40, COMPRESIÓN ALTA

In [1]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
from facenet_pytorch import MTCNN
from tqdm import tqdm


# --- RUTAS PARA C40 ---
BASE_PATH = r'C:\TFG\Dataset C40'
DESTINY_PATH = r'C:\TFG\Dataset_C40_Procesado'

os.makedirs(DESTINY_PATH, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mtcnn = MTCNN(image_size=299, margin=30, keep_all=False, post_process=False, device=device)

c:\TFG\venv_tfg\Lib\site-packages\facenet_pytorch\models\mtcnn.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(state_dict_path)
c:\TFG\venv_tfg\

In [2]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Trabajando con: {device}") 


Trabajando con: cuda


## Vemos cuantas carpets e imágenes tenemos

In [8]:
import os

# 1. PEGA TU RUTA AQUÍ
BASE_PATH = r'C:\TFG\Dataset_C40\FFPP_C40' 

def diagnostico_profundo(ruta):
    print(f"🔍 Analizando: {ruta}")
    if not os.path.exists(ruta):
        print("❌ ERROR: La ruta NO existe para Python. Revisa si hay una letra de unidad distinta (D:, E:, etc.)")
        return

    # Ver subcarpetas
    items = os.listdir(ruta)
    print(f"📁 Carpetas encontradas en la base: {items}")

    for root, dirs, files in os.walk(ruta):
        # Filtrar solo archivos de video
        videos = [f for f in files if f.lower().endswith(('.mp4', '.avi', '.mkv'))]
        
        if videos:
            nivel = root.replace(ruta, "")
            print(f"\n✅ Encontrada carpeta con vídeos: ...{nivel}")
            print(f"   Cant. vídeos: {len(videos)}")
            print(f"   Ejemplo de nombre: '{videos[0]}'")
            
            # Verificar si el nombre empieza por un número de 3 cifras
            ejemplo = videos[0][:3]
            if ejemplo.isdigit():
                print(f"   🔢 El nombre empieza por número: SÍ ({ejemplo})")
            else:
                print(f"   ❓ El nombre empieza por: '{ejemplo}' (Esto podría ser el problema)")

diagnostico_profundo(BASE_PATH)

🔍 Analizando: C:\TFG\Dataset_C40\FFPP_C40
📁 Carpetas encontradas en la base: ['Fake', 'Real']

✅ Encontrada carpeta con vídeos: ...\Fake\Deepfakes\videos
   Cant. vídeos: 1000
   Ejemplo de nombre: '000_003.mp4'
   🔢 El nombre empieza por número: SÍ (000)

✅ Encontrada carpeta con vídeos: ...\Fake\Face2Face\videos
   Cant. vídeos: 1000
   Ejemplo de nombre: '000_003.mp4'
   🔢 El nombre empieza por número: SÍ (000)

✅ Encontrada carpeta con vídeos: ...\Fake\FaceSwap\videos
   Cant. vídeos: 1000
   Ejemplo de nombre: '000_003.mp4'
   🔢 El nombre empieza por número: SÍ (000)

✅ Encontrada carpeta con vídeos: ...\Fake\NeuralTextures\videos
   Cant. vídeos: 1000
   Ejemplo de nombre: '000_003.mp4'
   🔢 El nombre empieza por número: SÍ (000)

✅ Encontrada carpeta con vídeos: ...\Real\youtube\videos
   Cant. vídeos: 1000
   Ejemplo de nombre: '000.mp4'
   🔢 El nombre empieza por número: SÍ (000)


In [1]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
from facenet_pytorch import MTCNN
from tqdm import tqdm

# ==========================================
# 1. CONFIGURACIÓN DE RUTAS LOCALES
# ==========================================
BASE_PATH = r'C:\TFG\Dataset_C40\FFPP_C40'
DESTINY_PATH = r'C:\TFG\Dataset_C40_Procesado'

FRAMES_PER_VIDEO = 15  # Igual que en tu C23
IMG_SIZE = 299        

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Dispositivo de procesamiento: {device}")

mtcnn = MTCNN(image_size=IMG_SIZE, margin=30, keep_all=False, post_process=False, device=device)

# ==========================================
# 2. GENERAR PLAN BALANCEADO (TEST SET)
# ==========================================
def generar_plan_balanceado():
    instrucciones = []
    
    # --- REALES (200 vídeos de Youtube) ---
    path_real = os.path.join(BASE_PATH, 'Real', 'youtube', 'videos')
    if os.path.exists(path_real):
        # Tomamos el rango 800-999 que es el estándar de test
        for v_id in range(800, 1000):
            v_name = f"{v_id:03d}.mp4"
            full_path = os.path.join(path_real, v_name)
            if os.path.exists(full_path):
                instrucciones.append({'path': full_path, 'label': 'real', 'tech': 'original', 'id': v_id})

    # --- FAKES (200 vídeos: 50 por técnica para balancear) ---
    tecnicas = ['Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures']
    
    for i, tech in enumerate(tecnicas):
        path_fake = os.path.join(BASE_PATH, 'Fake', tech, 'videos')
        if os.path.exists(path_fake):
            archivos = os.listdir(path_fake)
            # Cada técnica aporta 50 vídeos para sumar 200 fakes en total
            inicio = 800 + (i * 50)
            fin = inicio + 50
            
            for v_id in range(inicio, fin):
                # Buscamos el archivo que empiece por el ID (ej: 800_)
                match = [f for f in archivos if f.startswith(f"{v_id:03d}_") and f.endswith('.mp4')]
                if match:
                    instrucciones.append({
                        'path': os.path.join(path_fake, match[0]),
                        'label': 'fake', 'tech': tech, 'id': v_id
                    })

    return pd.DataFrame(instrucciones)

# ==========================================
# 3. EJECUCIÓN DE EXTRACCIÓN
# ==========================================
def ejecutar_extraccion(df):
    print("-" * 70)
    print(f"📊 RESUMEN DEL PLAN C40:")
    print(f"Total vídeos: {len(df)}")
    print(df.groupby('label').size())
    print("-" * 70)

    total_imgs = 0
    for idx, row in tqdm(df.iterrows(), total=df.shape[0]):
        # Carpeta: Dataset_C40_Procesado/test/real o /fake
        target_dir = os.path.join(DESTINY_PATH, 'test', row['label'])
        os.makedirs(target_dir, exist_ok=True)

        v_id_str = f"{row['id']:03d}"
        nombre_base = f"C40_{row['tech']}_{v_id_str}"

        # Saltar si ya está procesado
        if os.path.exists(os.path.join(target_dir, f"{nombre_base}_f14.jpg")):
            continue

        cap = cv2.VideoCapture(row['path'])
        total_f = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        if total_f < FRAMES_PER_VIDEO:
            cap.release()
            continue

        indices = np.linspace(0, total_f - 1, FRAMES_PER_VIDEO, dtype=int)
        
        for i_frame, pos in enumerate(indices):
            cap.set(cv2.CAP_PROP_POS_FRAMES, pos)
            ret, frame = cap.read()
            if not ret: break

            try:
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                face = mtcnn(frame_rgb)
                if face is not None:
                    face_img = face.permute(1, 2, 0).cpu().numpy().astype(np.uint8)
                    file_name = f"{nombre_base}_f{i_frame}.jpg"
                    cv2.imwrite(os.path.join(target_dir, file_name), 
                                cv2.cvtColor(face_img, cv2.COLOR_RGB2BGR))
                    total_imgs += 1
            except:
                continue
        cap.release()

    print(f"\n✨ PROCESO FINALIZADO. Total imágenes para test: {total_imgs}")

if __name__ == "__main__":
    df_test = generar_plan_balanceado()
    ejecutar_extraccion(df_test)

c:\TFG\venv_tfg\Lib\site-packages\facenet_pytorch\models\mtcnn.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(state_dict_path)
c:\TFG\venv_tfg\

✅ Dispositivo de procesamiento: cuda
----------------------------------------------------------------------
📊 RESUMEN DEL PLAN C40:
Total vídeos: 400
label
fake    200
real    200
dtype: int64
----------------------------------------------------------------------


100%|██████████| 400/400 [08:07<00:00,  1.22s/it]


✨ PROCESO FINALIZADO. Total imágenes para test: 5760


# Comprobación de dataset 
### Tiene que estar balanceado 50/50 imagenes reales/falsas. 

### Recortadas con MTCNN y comprobar que no haya data leakage

In [3]:
import os
import pandas as pd

# La ruta donde se guardaron las caras extraídas
PATH_PROCESADO = r'C:\TFG\Dataset_C40_Procesado\test'

def auditar_dataset_c40(ruta_base):
    print(f"🔍 Iniciando auditoría en: {ruta_base}")
    print("-" * 50)
    
    datos = []
    
    # Comprobar si la ruta existe
    if not os.path.exists(ruta_base):
        print(f"❌ ERROR: No existe la ruta {ruta_base}")
        return

    # Recorremos las carpetas real y fake
    for label in ['real', 'fake']:
        ruta_label = os.path.join(ruta_base, label)
        if not os.path.exists(ruta_label):
            print(f"⚠️ Aviso: No se encuentra la subcarpeta: {label}")
            continue
            
        archivos = [f for f in os.listdir(ruta_label) if f.endswith('.jpg')]
        
        for arch in archivos:
            # Ejemplo: C40_Deepfakes_800_f0.jpg
            partes = arch.replace('.jpg', '').split('_')
            try:
                v_id = partes[-2] # El ID suele ser el penúltimo
                tech = partes[1]  # La técnica suele ser el segundo
                datos.append({'archivo': arch, 'label': label, 'id': v_id, 'tech': tech})
            except:
                continue
    
    df = pd.DataFrame(datos)
    
    if df.empty:
        print("❌ El dataset está vacío. Verifica que se hayan guardado imágenes.")
        return

    # 1. BALANCEO DE IMÁGENES (CORREGIDO AQUÍ)
    print("📊 BALANCEO DE IMÁGENES:")
    conteo_labels = df['label'].value_counts()
    for lab, cant in conteo_labels.items():
        porcentaje = (cant / len(df)) * 100
        print(f"   - {lab.upper()}: {cant} imágenes ({porcentaje:.1f}%)")
    
    # 2. VÍDEOS ÚNICOS
    print("\n🎬 VÍDEOS ÚNICOS POR CLASE:")
    videos_unicos = df.groupby('label')['id'].nunique()
    for lab, cant in videos_unicos.items():
        print(f"   - {lab.upper()}: {cant} vídeos únicos")

    # 3. DATA LEAKAGE
    ids_reales = set(df[df['label'] == 'real']['id'])
    ids_fakes = set(df[df['label'] == 'fake']['id'])
    solapados = ids_reales.intersection(ids_fakes)
    
    print("\n🛡️ ANÁLISIS DE DATA LEAKAGE:")
    if len(solapados) == 0:
        print("   ✅ LIMPIO: No hay IDs compartidos.")
    else:
        print(f"   ℹ️ INFO: Hay {len(solapados)} IDs base compartidos (Normal en FF++ Test).")

    # 4. INTEGRIDAD
    print("\n📸 INTEGRIDAD DE FRAMES:")
    frames_por_video = df.groupby(['label', 'id']).size()
    incompletos = frames_por_video[frames_por_video < 15]
    if incompletos.empty:
        print("   ✅ PERFECTO: Todos los vídeos tienen 15 frames.")
    else:
        print(f"   ⚠️ AVISO: {len(incompletos)} vídeos tienen menos de 15 frames.")

# Ejecutar
auditar_dataset_c40(PATH_PROCESADO)

🔍 Iniciando auditoría en: C:\TFG\Dataset_C40_Procesado\test
--------------------------------------------------
📊 BALANCEO DE IMÁGENES:
   - FAKE: 3000 imágenes (50.0%)
   - REAL: 2999 imágenes (50.0%)

🎬 VÍDEOS ÚNICOS POR CLASE:
   - FAKE: 200 vídeos únicos
   - REAL: 200 vídeos únicos

🛡️ ANÁLISIS DE DATA LEAKAGE:
   ℹ️ INFO: Hay 200 IDs base compartidos (Normal en FF++ Test).

📸 INTEGRIDAD DE FRAMES:
   ⚠️ AVISO: 1 vídeos tienen menos de 15 frames.
